# 03 · Migración a LangGraph

**Bloque:** 0:45–1:10 · **Práctica:** 0:54–1:10

### Teoría breve
- **¿Por qué migrar?** Necesitamos **control explícito** del flujo, **estado** compartido, **ramas** y reintentos.
- **LangGraph:** orquesta agentes como **grafos**. *Nodos = pasos*, *edges (aristas) = transiciones*.

Construiremos un grafo que **clasifica** la pregunta y la enruta a un nodo técnico o general (**ramificación**).

In [ ]:
import sys, os
REPO = os.path.abspath("..")
if REPO not in sys.path:
    sys.path.insert(0, REPO)
from config import get_chat_model, get_embeddings
print("Entorno cargado ✔")

## Paso 1 · Definir el estado
El **estado** es el diccionario que fluye entre nodos.

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

class Estado(TypedDict):
    pregunta: str
    categoria: str
    respuesta: str

## Paso 2 · Definir los nodos
Cada nodo es una función que recibe el estado y devuelve los campos que actualiza.

In [ ]:
llm = get_chat_model()

def clasificar(estado: Estado):
    cat = llm.invoke(
        f"Clasifica en una sola palabra ('tecnica' o 'general') esta pregunta: {estado['pregunta']}"
    ).content.strip().lower()
    return {"categoria": "tecnica" if "tecn" in cat else "general"}

def responder_tecnico(estado: Estado):
    return {"respuesta": llm.invoke(f"Responde de forma técnica y precisa: {estado['pregunta']}").content}

def responder_general(estado: Estado):
    return {"respuesta": llm.invoke(f"Responde de forma simple, para principiantes: {estado['pregunta']}").content}

def ruta(estado: Estado):
    return estado["categoria"]  # 'tecnica' o 'general'

## Paso 3 · Construir el grafo
> **TODO:** agrega las **aristas condicionales** desde `clasificar` usando `add_conditional_edges`,
> enrutando `"tecnica"` → `responder_tecnico` y `"general"` → `responder_general`.

In [ ]:
g = StateGraph(Estado)
g.add_node("clasificar", clasificar)
g.add_node("responder_tecnico", responder_tecnico)
g.add_node("responder_general", responder_general)

g.add_edge(START, "clasificar")

# TODO: aristas condicionales según la categoría
____

g.add_edge("responder_tecnico", END)
g.add_edge("responder_general", END)

app = g.compile()
print("Grafo compilado ✔")

## Paso 4 · Visualizar y ejecutar

In [ ]:
print(app.get_graph().draw_mermaid())

In [ ]:
salida = app.invoke({"pregunta": "¿Qué es un embedding?", "categoria": "", "respuesta": ""})
print("Categoría detectada:", salida["categoria"])
print(salida["respuesta"])

## ✅ Checkpoint 3
Tienes un flujo de **3+ nodos con ramificación**. Prueba una pregunta 'general' y confirma que toma la otra rama.

In [ ]:
salida = app.invoke({"pregunta": "¿Para qué sirve la IA en la vida diaria?", "categoria": "", "respuesta": ""})
print("Categoría detectada:", salida["categoria"])
print(salida["respuesta"])